# 🚀 Training GO 1.0 (goo1) via Knowledge Distillation on Google Colab

**Project:** FRIDAY — Autonomous Local AI Assistant & Neural Lab  
**Student Model:** Google Gemma 2 2B (`google/gemma-2-2b-it`)  
**Technique:** Sequence-Level Knowledge Distillation + QLoRA (4-bit SFT)  
**Target Deliverable:** `goo1-Q4_K_M.gguf` for high-speed edge execution on Mac / FRIDAY HUD

---
### Overview of Steps:
1. **GPU Check & Package Setup** (Transformers, PEFT, TRL, bitsandbytes, accelerate)
2. **Hugging Face Authentication & Student Model Loading**
3. **Distilled Knowledge Dataset Preparation**
4. **QLoRA Hyperparameter Setup & Training Loop**
5. **Validation & Benchmark Prompts**
6. **LoRA Fusion & GGUF Quantization (Q4_K_M)**
7. **Google Drive Sync / Download to Local Mac**

## Step 1: Verify Colab GPU & Install Acceleration Suite
Make sure your Colab Runtime is set to **GPU** (`Runtime -> Change runtime type -> T4 GPU` or `A100`).

In [ ]:
import os

# Check allocated GPU
os.system("nvidia-smi")

# Install high-performance LLM training stack
os.system("pip install --quiet --upgrade pip")
os.system("pip install --quiet transformers peft trl bitsandbytes accelerate datasets sentencepiece")
print("\n[✓] Environment setup completed successfully.")

## Step 2: Hugging Face Authentication & Model Loading
Gemma 2 requires accepting the license on Hugging Face: [huggingface.co/google/gemma-2-2b-it](https://huggingface.co/google/gemma-2-2b-it).

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "unsloth/gemma-2-2b-it"
max_seq_length = 2048

# 4-bit Quantization Config to fit inside 15GB T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"[*] Loading student model: {model_id} (ungated) in 4-bit...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)
print("[✓] Student base model initialized into VRAM successfully!")


## Step 3: Apply LoRA Adapters for Distillation

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
print("[✓] LoRA parameter-efficient adapters configured.")

## Step 4: Prepare the Distilled Training Corpus
This dataset contains high-quality teacher responses covering: emails, translation, text analysis, general reasoning, and FRIDAY tool calls.

In [ ]:
import os
import json
from datasets import Dataset

dataset_file = "go1_distilled_corpus.jsonl"
records = []

if os.path.exists(dataset_file):
    with open(dataset_file, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    print(f"[✓] Loaded {len(records)} examples from uploaded file.")
else:
    # Fallback to rich distilled demonstration dataset
    records = [
        {
            "instruction": "Who are you and what are your capabilities?",
            "response": "I am GO 1.0 (goo1), a specialized AI language model trained on Google Gemma 2 (2B) using Knowledge Distillation. I excel at composing structured professional emails, translating between languages with cultural nuances, performing text analysis, writing clean code, and calling local FRIDAY tools."
        },
        {
            "instruction": "Draft a professional email proposing an on-device local AI assistant for company privacy.",
            "response": "Subject: Proposal: On-Device AI Infrastructure for Enhanced Data Privacy\n\nDear Executive Team,\n\nI am writing to propose transitioning our routine internal workflow automation to on-device local AI models.\n\nKey Benefits:\n• Absolute Data Privacy: Zero code or sensitive documentation leaves our local hardware.\n• Zero Recurring API Costs: Predictable infrastructure with no token rate limits.\n• Sub-Second Local Latency: Native Apple Silicon and GPU acceleration.\n\nI welcome the opportunity to present a live demonstration next week.\n\nSincerely,\n[Your Name]\nEngineering Lead"
        },
        {
            "instruction": "Translate to French, Spanish, and German: 'Knowledge distillation allows compact models to achieve frontier intelligence.'",
            "response": "**French:**\n« La distillation des connaissances permet aux modèles compacts d'atteindre une intelligence de pointe. »\n\n**Spanish:**\n« La destilación de conocimiento permite que los modelos compactos alcancen una inteligencia de vanguardia. »\n\n**German:**\n„Wissensdestillation ermöglicht es kompakten Modellen, Spitzenintelligenz zu erreichen.“"
        },
        {
            "instruction": "What is the current battery level and RAM usage of my Mac?",
            "response": "Checking your macOS hardware diagnostics:\n[TOOL_REQUEST: system_status()]"
        },
        {
            "instruction": "Calculate 125.5 multiplied by 48 divided by 1.2",
            "response": 'Computing with safe AST calculator:\n[TOOL_REQUEST: calculator(expression="(125.5 * 48) / 1.2")]'
        }
    ]
    print(f"[*] Initialized seed dataset with {len(records)} foundational pairs.")

# Format for Gemma 2 turn tokens
def format_turn(example):
    text = f"<start_of_turn>user\n{example['instruction']}<end_of_turn>\n<start_of_turn>model\n{example['response']}<end_of_turn>"
    return {"text": text}

raw_dataset = Dataset.from_list(records)
train_dataset = raw_dataset.map(format_turn)
print("[✓] Dataset formatted with Gemma 2 conversational tokens.")

## Step 5: Execute Student Training (SFTTrainer)

In [ ]:
import torch
from trl import SFTTrainer, SFTConfig

sft_args = SFTConfig(
    dataset_text_field="text",
    max_length=2048,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=40,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5,
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    output_dir="checkpoints_go1",
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    args=sft_args,
)

print("[*] Commencing student knowledge distillation training on Colab T4...")
trainer_stats = trainer.train()
print(f"\n[✓] Distillation training complete! Final Loss: {trainer_stats.training_loss:.4f}")


## Step 6: Test Model Inference Before Export

In [ ]:
model.eval()

test_prompt = "Write an email thanking a partner for a productive demo and proposing next steps for Friday."
inputs = tokenizer(
    [f"<start_of_turn>user\n{test_prompt}<end_of_turn>\n<start_of_turn>model\n"],
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.35)
result = tokenizer.decode(outputs[0], skip_special_tokens=False)
print("\n--- Model Output Test ---\n")
print(result)

## Step 7: LoRA Fusion & GGUF Quantization (Export to Mac)
This compiles the model into `goo1-Q4_K_M.gguf`, ready for direct local execution in Ollama!

In [ ]:
import os
import torch
from peft import PeftModel

print("[*] Saving trained LoRA adapter weights...")
model.save_pretrained("goo1_lora_adapter")
tokenizer.save_pretrained("goo1_lora_adapter")
print("[✓] Saved LoRA adapter to goo1_lora_adapter/")

# Setup llama.cpp for direct 4-bit GGUF conversion
print("[*] Setting up llama.cpp for GGUF compilation...")
if not os.path.exists("llama.cpp"):
    os.system("git clone --depth 1 https://github.com/ggerganov/llama.cpp")
    os.system("cd llama.cpp && make GGML_CUDA=1 -j4")
    os.system("pip install -r llama.cpp/requirements.txt")

# Merge adapter into base float16 model
print("[*] Merging LoRA adapters into base Gemma 2 weights...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="cpu",
)
merged_model = PeftModel.from_pretrained(base_model, "goo1_lora_adapter")
merged_model = merged_model.merge_and_unload()

os.makedirs("goo1_merged", exist_ok=True)
merged_model.save_pretrained("goo1_merged")
tokenizer.save_pretrained("goo1_merged")
print("[✓] Standalone merged model saved to goo1_merged/")

# Convert to GGUF
print("[*] Compiling to GGUF format...")
os.system("python3 llama.cpp/convert_hf_to_gguf.py goo1_merged --outtype f16 --outfile goo1_f16.gguf")
os.system("./llama.cpp/llama-quantize goo1_f16.gguf goo1-Q4_K_M.gguf q4_k_m")

print("\n[✓] SUCCESS: Created goo1-Q4_K_M.gguf ready for Mac execution!")

# Mount Google Drive for easy download
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.system("cp goo1-Q4_K_M.gguf /content/drive/MyDrive/ 2>/dev/null || cp *Q4_K_M.gguf /content/drive/MyDrive/")
    print("[✓] Model synced to your Google Drive: /content/drive/MyDrive/goo1-Q4_K_M.gguf")
except Exception as e:
    print(f"Drive note: {e}. You can download goo1-Q4_K_M.gguf directly from the Colab left sidebar.")